In [1]:
# %%
!pip install pyreadstat --break-system-packages

In [2]:
# %%
import pandas as pd
import pyreadstat
import warnings
warnings.filterwarnings('ignore')
import os
os.chdir(r"C:\Users\Hp\Downloads\Project 2026 DS")

In [3]:
# %%
# 1. CHECK COLUMN NAMES FOR 2016-17 FILE
_, meta16 = pyreadstat.read_sav(
    r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\surveydata1617.sav",
    metadataonly=True
)

cl_lbl16 = pd.DataFrame({
    'column': meta16.column_names,
    'label': meta16.column_labels
})
print(cl_lbl16.to_string())

                                             column                                                                                                                                                                                                                     label
0                                            serial                                                                                                                                                                                                                    Serial
1                                              Type                                                                                                                                                                                           Adult or Young Person interview
2                                              mode                                                                                                                                           

In [4]:
# %%
# 2. SEARCH FOR MSOA COLUMN (just a check, not used later unless found)
msoa_check = cl_lbl16[
    cl_lbl16['column'].str.lower().str.contains('msoa', na=False) |
    cl_lbl16['label'].str.lower().str.contains('msoa', na=False)
]
print(msoa_check.to_string())

Empty DataFrame
Columns: [column, label]
Index: []


In [5]:
# %%
# 3. LOAD 2016-17 DATA, ONLY THE COLUMNS WE NEED
cols_1617 = [
    'serial', 'wt_final', 'Reg9', 'LA', 'LondInOut',
    'Age9', 'Gend3', 'Eth7', 'IMD10', 'Disab3',
    'NSSEC5', 'Educ6', 'Orient4', 'Relig7',
    'ChildAgeU13', 'Maternity_pop',
    'Filter_Act', 'Filter_InsAct', 'Filter_Inact'
]

df16, meta16 = pyreadstat.read_sav(
    r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\surveydata1617.sav",
    usecols=cols_1617,
    apply_value_formats=True
)

print(df16.shape)
print(df16['Reg9'].value_counts())

(196635, 19)
Reg9
South East       36345
North West       27560
East             25806
South West       22349
East Midlands    22176
London           19497
West Midlands    19092
Yorkshire        14852
North East        8958
Name: count, dtype: int64


In [6]:
# %%
# 4. FILTER TO LONDON BOROUGHS ONLY (2016-17)
lon16 = df16[df16['LA'].str.startswith('E09')].copy()

if pd.api.types.is_categorical_dtype(lon16['LA']):
    lon16['LA'] = lon16['LA'].cat.remove_unused_categories()

print(f"London respondents 2016-17: {len(lon16)}")
print(f"Boroughs: {lon16['LA'].nunique()}")

London respondents 2016-17: 19497
Boroughs: 33


In [7]:
# %%
# CHECK Age9 BEFORE ANY FILLNA OR MERGE CODE RUNS
print(lon16['Age9'].unique())

['16-24', '35-44', '45-54', '55-64', '25-34', '65-74', '75-84', '85+', NaN]
Categories (8, object): ['16-24', '25-34', '35-44', '45-54', '55-64', '65-74', '75-84', '85+']


In [9]:
# %%
both_missing = lon16[lon16['Orient4'].isna() & lon16['Relig7'].isna()]
print(f"Missing both: {len(both_missing)}")

Missing both: 13950


In [10]:
# %%
# CHECK IF Orient4 and Relig7 MISSINGNESS OVERLAPS (same people not asked both)
both_missing = lon16[lon16['Orient4'].isna() & lon16['Relig7'].isna()]
print(f"Missing both: {len(both_missing)}")
print(f"Missing Orient4 only: {lon16['Orient4'].isna().sum() - len(both_missing)}")
print(f"Missing Relig7 only: {lon16['Relig7'].isna().sum() - len(both_missing)}")

Missing both: 13950
Missing Orient4 only: 221
Missing Relig7 only: 295


In [11]:
# %%
# CHECK IF "Not asked" IS REALLY UNDER Age9, OR A MISALIGNMENT IN THE STACKED FILE
age9_only = lon16.groupby(['LA', 'Age9']).apply(lambda x: pd.Series({
    'pct_active': (x['Filter_Act'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
    'respondents': len(x)
})).reset_index()

print(age9_only[age9_only['Age9'] == 'Not asked / Not applicable'])
print(age9_only['Age9'].unique())

Empty DataFrame
Columns: [LA, Age9, pct_active, respondents]
Index: []
['16-24', '25-34', '35-44', '45-54', '55-64', '65-74', '75-84', '85+']
Categories (8, object): ['16-24', '25-34', '35-44', '45-54', '55-64', '65-74', '75-84', '85+']


In [12]:
# %%
# 4b. CHECK DTYPES
print(lon16['Filter_Act'].dtype)
print(lon16['Filter_InsAct'].dtype)
print(lon16['Filter_Inact'].dtype)
print(lon16['wt_final'].dtype)

float64
float64
float64
float64


In [13]:
# %%
print(lon16['LA'].dtype)
print(lon16['IMD10'].dtype)
print(lon16['NSSEC5'].dtype)
print(lon16['Eth7'].dtype)

category
category
category
category


In [14]:
# %%
# 4d. CONVERT ALL RELEVANT COLUMNS TO PLAIN TYPES (2016-17) - FIXED VERSION
group_cols = ['LA', 'Age9', 'Gend3', 'Eth7', 'IMD10', 'Disab3',
              'LondInOut', 'NSSEC5', 'Educ6', 'Orient4', 'Relig7',
              'ChildAgeU13', 'Maternity_pop']

# convert out of Categorical first, then fill missing values, then make it plain text
for col in group_cols:
    lon16[col] = lon16[col].astype(object).fillna('Not asked / Not applicable').astype(str)

# fix IMD10 - reload it as raw numbers since the labels are broken for deciles 2-9
imd_raw16, _ = pyreadstat.read_sav(
    r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\surveydata1617.sav",
    usecols=['serial', 'IMD10'],
    apply_value_formats=False
)
imd_raw16 = imd_raw16.rename(columns={'IMD10': 'IMD10_raw'})

lon16 = lon16.merge(imd_raw16, on='serial', how='left')

imd_map = {i: f"Decile {i}" for i in range(1, 11)}
imd_map[1] = "Decile 1 (most deprived)"
imd_map[10] = "Decile 10 (least deprived)"

lon16['IMD10_numeric'] = lon16['IMD10_raw']
lon16['IMD10'] = lon16['IMD10_raw'].map(imd_map).fillna('Not asked / Not applicable')
lon16 = lon16.drop(columns=['IMD10_raw'])

lon16['Filter_Act'] = lon16['Filter_Act'].astype(float)
lon16['Filter_InsAct'] = lon16['Filter_InsAct'].astype(float)
lon16['Filter_Inact'] = lon16['Filter_Inact'].astype(float)
lon16['wt_final'] = lon16['wt_final'].astype(float)

print(lon16.dtypes)
print(lon16['IMD10'].value_counts())

serial            float64
wt_final          float64
Filter_Act        float64
Filter_InsAct     float64
Filter_Inact      float64
Reg9             category
LondInOut          object
LA                 object
Age9               object
ChildAgeU13        object
Maternity_pop      object
Disab3             object
Educ6              object
Eth7               object
Gend3              object
IMD10              object
NSSEC5             object
Orient4            object
Relig7             object
IMD10_numeric     float64
dtype: object
IMD10
Decile 2                      3868
Decile 3                      2843
Decile 4                      2286
Decile 1 (most deprived)      2272
Decile 5                      1870
Decile 6                      1790
Decile 7                      1475
Decile 9                      1296
Decile 8                      1225
Decile 10 (least deprived)     572
Name: count, dtype: int64


In [15]:
# %%
# 5. QUICK LOOK AT EACH DEMOGRAPHIC COLUMN (2016-17)
check_cols = ['Age9', 'Gend3', 'Eth7', 'IMD10', 'Disab3', 'NSSEC5',
              'Educ6', 'Orient4', 'Relig7', 'ChildAgeU13', 'Maternity_pop']

for col in check_cols:
    print(f"--- {col} ---")
    print(lon16[col].value_counts())
    print()

--- Age9 ---
Age9
35-44                         4045
25-34                         3589
45-54                         3337
55-64                         2856
65-74                         2516
16-24                         1583
75-84                         1028
85+                            303
Not asked / Not applicable     240
Name: count, dtype: int64

--- Gend3 ---
Gend3
Female                        11085
Male                           8366
Not asked / Not applicable       45
Other                             1
Name: count, dtype: int64

--- Eth7 ---
Eth7
White British                 10250
White Other                    2845
South Asian                    2363
Not asked / Not applicable     1379
Black                          1302
Mixed                           524
Other ethnic group              488
Chinese                         346
Name: count, dtype: int64

--- IMD10 ---
IMD10
Decile 2                      3868
Decile 3                      2843
Decile 4                  

In [16]:
# %%
# 6. FUNCTION TO CALCULATE WEIGHTED ACTIVITY PERCENTAGES
def weighted_activity(df, group_col):
    result = df.groupby(group_col).apply(lambda x: pd.Series({
        'pct_active': (x['Filter_Act'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'pct_fairly_active': (x['Filter_InsAct'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'pct_inactive': (x['Filter_Inact'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'respondents': len(x),
        'weighted_base': x['wt_final'].sum()
    })).reset_index()
    return result

In [17]:
# %%
# 7. FILE 1 FOR 2016-17 - BOROUGH GAP SCORE TABLE
# one row per borough, this is the file that joins with the OpenActive map data
gap16 = weighted_activity(lon16, 'LA')
gap16 = gap16.rename(columns={'LA': 'borough'})
gap16['survey_year'] = '2016-17'

gap16 = gap16[['survey_year', 'borough', 'pct_active', 'pct_fairly_active',
               'pct_inactive', 'respondents', 'weighted_base']]

print(gap16.to_string())

   survey_year                           borough  pct_active  pct_fairly_active  pct_inactive  respondents  weighted_base
0      2016-17          E09000001 City of London   68.432196          14.591004     16.976800        249.0      36.588025
1      2016-17    E09000002 Barking and Dagenham   49.230649          16.065856     34.703494        958.0     664.418228
2      2016-17                  E09000003 Barnet   56.778087          12.280310     30.941603        997.0    1324.609800
3      2016-17                  E09000004 Bexley   56.715084          17.347982     25.936934        498.0     869.833270
4      2016-17                   E09000005 Brent   55.859792          11.182239     32.957969        489.0    1142.779105
5      2016-17                 E09000006 Bromley   69.016348          11.295237     19.688415        492.0    1137.321707
6      2016-17                  E09000007 Camden   72.092576          12.393416     15.514007        492.0     890.668723
7      2016-17          

In [18]:
# %%
# 8. FILE 2 FOR 2016-17 - POPULATION PROFILE TABLE
# long format, one row per borough x demographic group x category
demo_cols = ['Age9', 'Gend3', 'Eth7', 'IMD10', 'Disab3', 'LondInOut',
             'NSSEC5', 'Educ6', 'Orient4', 'Relig7', 'ChildAgeU13', 'Maternity_pop']

rows16 = []

for col in demo_cols:
    temp = lon16.groupby(['LA', col]).apply(lambda x: pd.Series({
        'pct_active': (x['Filter_Act'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'pct_fairly_active': (x['Filter_InsAct'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'pct_inactive': (x['Filter_Inact'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'respondents': len(x),
        'weighted_base': x['wt_final'].sum()
    })).reset_index()

    temp = temp.rename(columns={'LA': 'borough', col: 'category'})
    temp['demographic_group'] = col
    rows16.append(temp)

profile16 = pd.concat(rows16, ignore_index=True)
profile16['survey_year'] = '2016-17'
profile16['suppress'] = profile16['respondents'] < 30

profile16 = profile16[['survey_year', 'borough', 'demographic_group', 'category',
                        'pct_active', 'pct_fairly_active', 'pct_inactive',
                        'respondents', 'weighted_base', 'suppress']]

print(profile16.head(20).to_string())

   survey_year                         borough demographic_group                    category  pct_active  pct_fairly_active  pct_inactive  respondents  weighted_base  suppress
0      2016-17        E09000001 City of London              Age9                       16-24   51.129022          38.393983     10.476996         15.0       3.853513      True
1      2016-17        E09000001 City of London              Age9                       25-34   75.139492           9.568362     15.292146         44.0       8.975523     False
2      2016-17        E09000001 City of London              Age9                       35-44   84.271608           4.684610     11.043782         40.0       6.378827     False
3      2016-17        E09000001 City of London              Age9                       45-54   64.446569          17.074741     18.478690         46.0       5.282475     False
4      2016-17        E09000001 City of London              Age9                       55-64   84.058502           8.229

In [19]:
# %%
# 9. CHECK COLUMN NAMES FOR 2017-18 FILE
_, meta18 = pyreadstat.read_sav(
    r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\surveydata1718.sav",
    metadataonly=True
)

cl_lbl18 = pd.DataFrame({
    'column': meta18.column_names,
    'label': meta18.column_labels
})
print(cl_lbl18.to_string())

                                             column                                                                                                                                                                                                                     label
0                                            serial                                                                                                                                                                                                                    Serial
1                                              Type                                                                                                                                                                                                                      Type
2                                              mode                                                                                                                                           

In [20]:
# %%
# 10. LOAD 2017-18 DATA
# check the printout above matches cols_1617 names, edit list below if anything differs
cols_1718 = [
    'serial', 'wt_final', 'Reg9', 'LA', 'LondInOut',
    'Age9', 'Gend3', 'Eth7', 'IMD10', 'Disab3',
    'NSSEC5', 'Educ6', 'Orient4', 'Relig7',
    'ChildAgeU13', 'Maternity_pop',
    'Filter_Act', 'Filter_InsAct', 'Filter_Inact'
]

df18, meta18 = pyreadstat.read_sav(
    r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\surveydata1718.sav",
    usecols=cols_1718,
    apply_value_formats=True
)

print(df18.shape)
print(df18['Reg9'].value_counts())

(179747, 19)
Reg9
South East                  33665
North West                  27031
East                        23735
East Midlands               21557
South West                  19846
West Midlands               16561
London                      16200
Yorkshire and the Humber    13911
North East                   7241
Name: count, dtype: int64


In [21]:
# %%
# 11. FILTER TO LONDON BOROUGHS ONLY (2017-18)
lon18 = df18[df18['LA'].str.startswith('E09')].copy()

if pd.api.types.is_categorical_dtype(lon18['LA']):
    lon18['LA'] = lon18['LA'].cat.remove_unused_categories()

print(f"London respondents 2017-18: {len(lon18)}")
print(f"Boroughs: {lon18['LA'].nunique()}")

London respondents 2017-18: 16200
Boroughs: 33


In [22]:
# %%
group_cols = ['LA', 'Age9', 'Gend3', 'Eth7', 'IMD10', 'Disab3',
              'LondInOut', 'NSSEC5', 'Educ6', 'Orient4', 'Relig7',
              'ChildAgeU13', 'Maternity_pop']

for col in group_cols:
    n_missing = lon18[col].isna().sum()
    print(f"{col}: {n_missing} missing")

LA: 0 missing
Age9: 136 missing
Gend3: 35 missing
Eth7: 1141 missing
IMD10: 0 missing
Disab3: 1084 missing
LondInOut: 0 missing
NSSEC5: 1149 missing
Educ6: 598 missing
Orient4: 11210 missing
Relig7: 11324 missing
ChildAgeU13: 0 missing
Maternity_pop: 0 missing


In [23]:
# %%
# 11d. CONVERT ALL RELEVANT COLUMNS TO PLAIN TYPES (2017-18) - FIXED VERSION
for col in group_cols:
    lon18[col] = lon18[col].astype(object).fillna('Not asked / Not applicable').astype(str)

imd_raw18, _ = pyreadstat.read_sav(
    r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\surveydata1718.sav",
    usecols=['serial', 'IMD10'],
    apply_value_formats=False
)
imd_raw18 = imd_raw18.rename(columns={'IMD10': 'IMD10_raw'})

lon18 = lon18.merge(imd_raw18, on='serial', how='left')

lon18['IMD10_numeric'] = lon18['IMD10_raw']
lon18['IMD10'] = lon18['IMD10_raw'].map(imd_map).fillna('Not asked / Not applicable')
lon18 = lon18.drop(columns=['IMD10_raw'])

lon18['Filter_Act'] = lon18['Filter_Act'].astype(float)
lon18['Filter_InsAct'] = lon18['Filter_InsAct'].astype(float)
lon18['Filter_Inact'] = lon18['Filter_Inact'].astype(float)
lon18['wt_final'] = lon18['wt_final'].astype(float)

print(lon18.dtypes)
print(lon18['IMD10'].value_counts())

serial            float64
wt_final          float64
Filter_Act        float64
Filter_InsAct     float64
Filter_Inact      float64
Reg9             category
LondInOut          object
LA                 object
Age9               object
ChildAgeU13        object
Maternity_pop      object
Disab3             object
Educ6              object
Eth7               object
Gend3              object
IMD10              object
NSSEC5             object
Orient4            object
Relig7             object
IMD10_numeric     float64
dtype: object
IMD10
Decile 2                      3194
Decile 3                      2315
Decile 1 (most deprived)      1928
Decile 4                      1840
Decile 5                      1585
Decile 6                      1413
Decile 7                      1207
Decile 9                      1194
Decile 8                      1001
Decile 10 (least deprived)     523
Name: count, dtype: int64


In [24]:
# %%
# 12. FILE 1 FOR 2017-18 - BOROUGH GAP SCORE TABLE
gap18 = weighted_activity(lon18, 'LA')
gap18 = gap18.rename(columns={'LA': 'borough'})
gap18['survey_year'] = '2017-18'

gap18 = gap18[['survey_year', 'borough', 'pct_active', 'pct_fairly_active',
               'pct_inactive', 'respondents', 'weighted_base']]

print(gap18.to_string())

   survey_year                           borough  pct_active  pct_fairly_active  pct_inactive  respondents  weighted_base
0      2017-18          E09000001 City of London   72.968164           9.479776     17.552060        233.0      26.640442
1      2017-18    E09000002 Barking and Dagenham   53.636881          10.156971     36.206148        472.0     621.662376
2      2017-18                  E09000003 Barnet   64.490549          12.540142     22.969308        494.0    1210.996717
3      2017-18                  E09000004 Bexley   68.922750           9.451319     21.625930        489.0     789.882393
4      2017-18                   E09000005 Brent   59.644566          10.554171     29.801263        506.0    1032.843844
5      2017-18                 E09000006 Bromley   63.776192          12.818702     23.405106        484.0    1057.978097
6      2017-18                  E09000007 Camden   70.656281          12.473074     16.870645        513.0     849.168396
7      2017-18          

In [25]:
# %%
# 13. FILE 2 FOR 2017-18 - POPULATION PROFILE TABLE
rows18 = []

for col in demo_cols:
    temp = lon18.groupby(['LA', col]).apply(lambda x: pd.Series({
        'pct_active': (x['Filter_Act'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'pct_fairly_active': (x['Filter_InsAct'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'pct_inactive': (x['Filter_Inact'] * x['wt_final']).sum() / x['wt_final'].sum() * 100,
        'respondents': len(x),
        'weighted_base': x['wt_final'].sum()
    })).reset_index()

    temp = temp.rename(columns={'LA': 'borough', col: 'category'})
    temp['demographic_group'] = col
    rows18.append(temp)

profile18 = pd.concat(rows18, ignore_index=True)
profile18['survey_year'] = '2017-18'
profile18['suppress'] = profile18['respondents'] < 30

profile18 = profile18[['survey_year', 'borough', 'demographic_group', 'category',
                        'pct_active', 'pct_fairly_active', 'pct_inactive',
                        'respondents', 'weighted_base', 'suppress']]

print(profile18.head(20).to_string())

   survey_year                         borough demographic_group                    category  pct_active  pct_fairly_active  pct_inactive  respondents  weighted_base  suppress
0      2017-18        E09000001 City of London              Age9                       16-24   61.381257           0.000000     38.618743          8.0       2.060656      True
1      2017-18        E09000001 City of London              Age9                       25-34   81.866356           6.097080     12.036564         35.0       6.906960     False
2      2017-18        E09000001 City of London              Age9                       35-44   86.983968           1.787016     11.229016         37.0       3.421066     False
3      2017-18        E09000001 City of London              Age9                       45-54   76.762415          16.818537      6.419048         39.0       3.926601     False
4      2017-18        E09000001 City of London              Age9                       55-64   72.319885          12.818

In [26]:
# %%
# 14. COMBINE BOTH YEARS AND SAVE THE FINAL TWO FILES
gap_final = pd.concat([gap16, gap18], ignore_index=True)
profile_final = pd.concat([profile16, profile18], ignore_index=True)

gap_final.to_csv(
    r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\gapscore.csv",
    index=False
)

profile_final.to_csv(
    r"C:\Users\Hp\Downloads\Project 2026 DS\ActiveLives_Data\popl_profile.csv",
    index=False
)

print("Saved 2 files:")
print(f"Gap score table: {gap_final.shape}")
print(f"Population profile table: {profile_final.shape}")

Saved 2 files:
Gap score table: (66, 7)
Population profile table: (4068, 10)
